# Example notebook for customizing circuit with new e-models

This notebook demonstrates how to create a customized circuit by:
1. Downloading a parent circuit from entitycore
2. Adding new e-models (hoc files) and modifying the nodes file accordingly
3. Validating the modifications, creating the modified circuit and uploading it in a single function

## Setup

In [1]:
import h5py
from pathlib import Path

from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

from obi_one.utils.circuit_customization.new_emodels import run

In [2]:
# Disable info logging (optional)
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# "Shared results + tests"
virtual_lab_id="e6030ed8-a589-4be2-80a6-f975406eb1f6"
project_id="2720f785-a3a2-4472-969d-19a53891c817"
token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
client = Client(environment="staging", project_context=project_context, token_manager=token)


## Configuration

Specify the parent circuit ID and output path for the customized circuit.
E-models compatible with the parent circuit are already present in the examples data file. In this case, the hoc from the circuit have been taken, and their conductance values have been changed. You can provide your own e-models if you want, but they have to use the mechanisms already present in the circuit.
The hoc files can be renamed, but that will imply to also update the nodes file.

In [4]:
# Parent circuit ID (from entitycore)
PARENT_CIRCUIT_ID = "0dcaa1ff-6d8c-43aa-a4a2-aa65ab706095"  # "nbS1-O1-E2PV-maxNsyn-HEX0-L2"

# Custom input file path
CUSTOM_FILE_PATH = Path("./custom_files")

# path with modified e-models
MODIFIED_EMODELS_DIR = Path("../data/modified_emodels")
MODIFIED_EMODELS_PATHS = [
    MODIFIED_EMODELS_DIR / "cACint_L23MC.hoc",
    MODIFIED_EMODELS_DIR / "cADpyr_L2TPC_v2.hoc"
]


# Output path for the new customized circuit
NEW_CIRCUIT_PATH = Path("./customized_circuit")

In [ ]:
# Print parent circuit information
parent_circuit = client.get_entity(entity_id=PARENT_CIRCUIT_ID, entity_type=models.Circuit)
print(f"{parent_circuit.name} (ID {parent_circuit.id})")
print(parent_circuit.description)
print(f"#Neurons: {parent_circuit.number_neurons}, #Synapses: {parent_circuit.number_synapses}, #Connections: {parent_circuit.number_connections}, Scale: {parent_circuit.scale}")

## Updating the nodes file

A hoc file with a new name has been provided in `MODIFIED_EMODELS_PATHS`: `cADpyr_L2TPC_v2.hoc`, so the nodes file has to be updated accordingly.

We need to
- download the nodes file
- update the model_template in the nodes file if the hoc file name(s) is different from the one(s) from the circuit
- update the etype in the nodes file if the new e-model(s) is from another etype

In [ ]:
# Download existing circuit config
from obi_one.utils.circuit_customization.download import download_node_populations

# Here we have to give the name of the population with biophysical nodes, in case there are multiple populations
nodes_paths = download_node_populations(PARENT_CIRCUIT_ID, client, CUSTOM_FILE_PATH, "S1nonbarrel_neurons")
nodes_path = nodes_paths[0]
print(f"Nodes file path downloaded to: {nodes_path}")


Here, we only modify cADpyr-related hoc names and etypes, because the other hoc has kept the same name (and the same etype for simplicity).

Note that the internal paths to the model_template and etype can be different for other nodes files, so modify them accordingly.

The name of the e-model should be in the shape `hoc:{file_name}`

In [ ]:
modified_hoc_ids = [1]
modified_hoc_names = [b"hoc:cADpyr_L2TPC_v2"]
modified_hoc_etypes = [b"special_cADpyr"]
with h5py.File(nodes_path,'a') as nodes:
    for id_, name, etype in zip(modified_hoc_ids, modified_hoc_names, modified_hoc_etypes):
        nodes["nodes/S1nonbarrel_neurons/0/model_template"][id_] = name
        nodes["nodes/S1nonbarrel_neurons/0/etype"][id_] = etype

## Validate modifications, Create modified circuit, and upload it

This can all be done with one single function. It will create the circuit in a new path. Make sure that the new path does not exist yet before running the function.

The validations include:
- checking that the hoc files are valid hoc templates
- checking that the new hoc files are not using mechanisms that are not present in the circuit
- checking that only model_template and/or etype columns have been changed in the new nodes file
- checking that all hoc files used in the new nodes file exist, either from the new hoc e-models, or from the existing circuit e-models
- checking that all new hoc file are used in the new nodes file
- checking that the hoc files can be initialized by bluecellulab. This ensure that they have the expected structure and that they can be run by our platform.

The circuit creation include:
- replacing the old nodes file with the new nodes file
- adding new e-models to the circuit e-models and removing the ones that are not used by the circuit anymore
- re-computing the dynamics params, i.e. the holding current, the resting potential, the input resistance and the threshold current
- re-writing me-combos

In [ ]:
modified_circuit_name = "nbS1-O1-E2PV-maxNsyn-HEX0-L2-modified"
modified_circuit_description = "nbS1-O1-E2PV-maxNsyn-HEX0-L2 with reduced sodium in soma and axon"
run(
    client=client,
    parent_circuit_id=PARENT_CIRCUIT_ID,
    new_node_file_path=nodes_path,
    new_hoc_files_paths=MODIFIED_EMODELS_PATHS,
    name=modified_circuit_name,
    description=modified_circuit_description,
    new_circuit_path=NEW_CIRCUIT_PATH,
    contact_email=None,
    dry_run=True,
)